# Traffic Sign Visibility Audit (GTSRB)
**Question:** which images should a human review team look at first, and what measured evidence supports that priority?
No model is trained: the workflow is OpenCV measurements -> NumPy percentile rule -> pandas class comparison -> Matplotlib figures.
Run all cells from top to bottom (Kernel > Restart & Run All). Dataset: GTSRB training images (see README for the citation and download).

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import traffic_audit as ta
from traffic_audit import TrafficSignAudit

# ---- settings (edit here; relative paths only) ----
DATA_DIR = ROOT / "data"   # for the synthetic smoke test use: ROOT/"demo"/"data"/"Final_Training"/"Images"
OUT_DIR  = ROOT / "outputs"                                          # smoke test: ROOT/"demo"/"outputs"
CLASSES  = (4, 11, 33, 36, 40)                         # my selected classes (see README / notes)
CAP, SEED = 500, 42
audit = TrafficSignAudit(DATA_DIR, OUT_DIR, CLASSES, per_class_cap=CAP, seed=SEED)
print("Demo/synthetic data?", audit.demo)

## Part 2 - Load and inspect the full annotations (full-dataset counts)

In [ ]:
annotations = audit.load_annotations()
rep = audit.report
print({k: v for k, v in rep.items() if k not in ("full_class_counts", "dtypes_raw")})
print("dtypes:", rep["dtypes_raw"])
pd.DataFrame(audit.skipped).groupby(["stage", "reason"]).size().rename("n").reset_index()

In [ ]:
full_counts = pd.Series(rep["full_class_counts"]).rename("n_full").to_frame()
full_counts["name"] = [ta.CLASS_NAMES[i] for i in full_counts.index]
full_counts.loc[list(CLASSES)].sort_values("n_full", ascending=False)   # FULL-dataset counts of the selected classes

## Part 3 - OpenCV measurements on a seeded stratified sample\nBrightness = mean grey level, contrast = grey-level std, sharpness = variance of the Laplacian (a *proxy*, not a readability score), min_side = shorter ROI side.

In [ ]:
table = audit.extract_features()
print(table.shape, "| unreadable:", (table["status"] != "ok").sum(), "| ROI fallback:", (~table["roi_ok_annotation"]).sum())
table[["image_path","class_id","width_ann","height_ann","roi_used","brightness","contrast","sharpness","min_side","status"]].head()

In [ ]:
# image-loading check + BGR -> RGB for display
row = table[table["status"] == "ok"].iloc[0]
bgr = cv2.imread(str(DATA_DIR / row["image_path"]))
assert bgr is not None
plt.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); plt.title(f"{row['image_path']} (RGB display of a BGR image)"); plt.axis("off"); plt.show()
audit.roi_vs_full()

## Part 4 - Review-priority rule (NumPy) and sensitivity check

In [ ]:
flagged = audit.assign_review_priority(pct=10, min_triggers=2)
print("Cutoffs (project-specific triage settings, NOT safety standards):", audit.cutoffs)
print(flagged["priority_tier"].value_counts())
rates, agree, top3, overall = audit.sensitivity_summary()
print("Flagged overall per setting:", overall); print("Rank agreement with baseline:", agree); print("Top-3:", top3)
rates.round(1)

## Part 5 - pandas summaries

In [ ]:
summary = audit.class_summary(); summary.round(2)

In [ ]:
print(audit.feature_differences().round(2))
print(audit.association().round(2))

## Figures (also written to `outputs/figures/`)

In [ ]:
audit.save_tables()
captions = audit.plot_all()
from IPython.display import Image, display
for f in sorted(audit.fig_dir.glob("fig0*.png")):
    display(Image(filename=str(f), width=850))

## Validation checks (required)

In [ ]:
import tempfile
from traffic_audit import ImageReadError, extract_image_features

# 1) an image-read failure is caught
with tempfile.TemporaryDirectory() as d:
    bad = Path(d) / "broken.ppm"; bad.write_bytes(b"not an image")
    try:
        extract_image_features(bad); raise AssertionError("no error raised")
    except ImageReadError as e:
        print("PASS read failure caught:", e)
# 2) output feature columns exist
need = {"brightness", "contrast", "sharpness", "min_side", "brightness_full", "contrast_full", "sharpness_full"}
assert need <= set(audit.flagged.columns); print("PASS feature columns present")
# 3) review flags contain only True/False
assert audit.flagged["review_flag"].dtype == bool and set(audit.flagged["review_flag"].unique()) <= {True, False}
print("PASS review_flag is boolean only")

## My interpretation (write in your own words)
*Edit this cell after looking at the figures: which classes stand out, which flagged images look like false alarms, which unflagged ones look like misses, and what you would check next. Then copy the key points into `report/student_notes.md`.*

In [ ]:
audit.run_all()   # recreates all tables, figures and results.json from the raw annotations in one call
print("Done - tables in", audit.proc_dir.relative_to(ROOT), "and figures in", audit.fig_dir.relative_to(ROOT))